[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/30.%20The%20Yard%20Pre-Marshalling%20for%20Exports%20Problem/P30-Tier-2_executed.ipynb)

---



# 30. The Yard Pre-Marshalling for Exports Problem
## Tier 2: List Scheduling Algorithm

### Goal and inputs

This notebook implements a list-scheduling heuristic for yard pre-marshalling. It treats containers as items to move between stacks and repeatedly selects the feasible move with the highest score. The score combines the immediate change in priority violations, stack balance, move efficiency, and the moved container's priority.

The input is a bay configuration: the number of stacks and tiers, and each container's priority and current stack and tier. The output is the resulting configuration, the sequence of moves selected by the heuristic, and a log of its progress.

### Scope and model assumptions

- Stack and tier indices start at zero. Tier 0 is the bottom position; the largest occupied tier is the top position.
- A stack is compact: its occupied tiers are consecutive, starting at tier 0. Each stack has capacity `tiers`.
- Only the top container of a stack can be moved. A move places it on top of a different stack that has available capacity.
- A priority violation is an adjacent inverted pair in a stack: a lower container has a larger priority number than the container immediately above it. Smaller priority numbers mean higher priority.
- The heuristic does not model travel time, crane interference, container retrieval dates beyond their priority order, or other operational constraints. Its score is a rule for ranking candidate moves, not a physical cost or an optimality certificate.

### How the method works

At each iteration, the algorithm enumerates feasible top-container moves, scores them, and executes the highest-scoring move. It stops when there are no priority violations, when there is no feasible move leading to a configuration not already visited, or when the move limit is reached. Remembering visited configurations prevents the heuristic from repeating a cycle. The method is a heuristic: it does not guarantee a globally optimal plan or that every instance can be solved within the move limit.

The concrete example has three stacks and the following initial configuration:

- Stack 0: bottom to top, B (priority 2), E (priority 5)
- Stack 1: bottom to top, D (priority 4), A (priority 1)
- Stack 2: bottom to top, C (priority 3)

The adjacent pair D below A is a priority violation because $4 > 1$. The pair B below E is not a violation because $2 < 5$.

In [ ]:
from dataclasses import dataclass
from typing import List, Optional, Tuple

import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

plt.style.use("default")

In [ ]:
@dataclass
class Container:
    """A container with a priority and a position in the bay."""

    id: str
    priority: int  # Lower values indicate higher priority.
    current_stack: int
    current_tier: int


@dataclass
class Move:
    """A move of one top container to another stack."""

    container_id: str
    from_stack: int
    from_tier: int
    to_stack: int
    to_tier: int
    score: float

In [ ]:
def validate_configuration(
    containers: List[Container],
    stacks: int,
    tiers: int,
) -> None:
    """Check that a bay state satisfies the model's basic physical assumptions."""
    if stacks <= 0 or tiers <= 0:
        raise ValueError("The numbers of stacks and tiers must be positive.")

    container_ids = [container.id for container in containers]
    if len(container_ids) != len(set(container_ids)):
        raise ValueError("Container IDs must be unique.")

    positions = set()
    for container in containers:
        if not 0 <= container.current_stack < stacks:
            raise ValueError(f"Container {container.id} has an invalid stack index.")
        if not 0 <= container.current_tier < tiers:
            raise ValueError(f"Container {container.id} has an invalid tier index.")

        position = (container.current_stack, container.current_tier)
        if position in positions:
            raise ValueError("Two containers cannot occupy the same position.")
        positions.add(position)

    for stack in range(stacks):
        occupied_tiers = sorted(
            container.current_tier
            for container in containers
            if container.current_stack == stack
        )
        if occupied_tiers != list(range(len(occupied_tiers))):
            raise ValueError(
                f"Stack {stack} must have consecutive occupied tiers starting at 0."
            )


def configuration_key(containers: List[Container]) -> Tuple[Tuple[str, int, int], ...]:
    """Return a stable representation for detecting repeated bay configurations."""
    return tuple(
        sorted(
            (container.id, container.current_stack, container.current_tier)
            for container in containers
        )
    )


def copy_containers(containers: List[Container]) -> List[Container]:
    """Copy container records so that a simulated move leaves its input unchanged."""
    return [
        Container(
            container.id,
            container.priority,
            container.current_stack,
            container.current_tier,
        )
        for container in containers
    ]

In [ ]:
class ListScheduler:
    """List-scheduling heuristic for the yard pre-marshalling problem."""

    def __init__(self, stacks: int, tiers: int):
        if stacks <= 0 or tiers <= 0:
            raise ValueError("The numbers of stacks and tiers must be positive.")

        self.stacks = stacks
        self.tiers = tiers
        self.move_history: List[Move] = []
        self.execution_log: List[str] = []

    def get_stack_height(self, containers: List[Container], stack: int) -> int:
        """Return the number of containers currently assigned to a stack."""
        return sum(container.current_stack == stack for container in containers)

    def get_top_container(
        self,
        containers: List[Container],
        stack: int,
    ) -> Optional[Container]:
        """Return the top container, or None if the stack is empty."""
        stack_containers = [
            container
            for container in containers
            if container.current_stack == stack
        ]
        if not stack_containers:
            return None
        return max(stack_containers, key=lambda container: container.current_tier)

    def count_priority_violations(self, containers: List[Container]) -> int:
        """Count adjacent priority inversions within all stacks."""
        violations = 0
        for stack in range(self.stacks):
            stack_containers = sorted(
                (
                    container
                    for container in containers
                    if container.current_stack == stack
                ),
                key=lambda container: container.current_tier,
            )
            for lower, upper in zip(stack_containers, stack_containers[1:]):
                if lower.priority > upper.priority:
                    violations += 1
        return violations

    def calculate_move_score(
        self,
        containers: List[Container],
        move: Move,
    ) -> float:
        """Score a candidate using its immediate effect and fixed heuristic weights."""
        container = next(
            item for item in containers if item.id == move.container_id
        )

        violations_before = self.count_priority_violations(containers)
        moved_configuration = self.execute_move(containers, move)
        violations_after = self.count_priority_violations(moved_configuration)
        violation_reduction = violations_before - violations_after
        score = 50 * violation_reduction

        heights = [
            self.get_stack_height(containers, stack)
            for stack in range(self.stacks)
        ]
        average_height = sum(heights) / self.stacks
        from_height = heights[move.from_stack]
        to_height = heights[move.to_stack]

        balance_before = (
            abs(from_height - average_height)
            + abs(to_height - average_height)
        )
        balance_after = (
            abs(from_height - 1 - average_height)
            + abs(to_height + 1 - average_height)
        )
        balance_improvement = balance_before - balance_after
        score += 20 * balance_improvement

        stack_distance = abs(move.from_stack - move.to_stack)
        height_penalty = self.tiers - 1 - move.from_tier
        efficiency = 15 - 2 * stack_distance - height_penalty
        score += max(0, efficiency)

        priority_bonus = 15 * (10 - container.priority) / 10
        score += priority_bonus
        return score

    def generate_valid_moves(self, containers: List[Container]) -> List[Move]:
        """Enumerate feasible moves of top containers to other non-full stacks."""
        moves = []
        for stack in range(self.stacks):
            top_container = self.get_top_container(containers, stack)
            if top_container is None:
                continue

            for target_stack in range(self.stacks):
                if target_stack == stack:
                    continue

                target_height = self.get_stack_height(containers, target_stack)
                if target_height >= self.tiers:
                    continue

                move = Move(
                    container_id=top_container.id,
                    from_stack=stack,
                    from_tier=top_container.current_tier,
                    to_stack=target_stack,
                    to_tier=target_height,
                    score=0.0,
                )
                move.score = self.calculate_move_score(containers, move)
                moves.append(move)
        return moves

    def execute_move(
        self,
        containers: List[Container],
        move: Move,
    ) -> List[Container]:
        """Apply a feasible move to a copy of the configuration."""
        moved_container = next(
            (item for item in containers if item.id == move.container_id),
            None,
        )
        if moved_container is None:
            raise ValueError(f"Container {move.container_id} is not in the bay.")

        top_container = self.get_top_container(containers, move.from_stack)
        target_height = self.get_stack_height(containers, move.to_stack)
        if (
            top_container is None
            or top_container.id != move.container_id
            or moved_container.current_tier != move.from_tier
            or move.from_stack == move.to_stack
            or move.to_tier != target_height
            or target_height >= self.tiers
        ):
            raise ValueError("The requested move is not feasible in this configuration.")

        new_containers = copy_containers(containers)
        for container in new_containers:
            if container.id == move.container_id:
                container.current_stack = move.to_stack
                container.current_tier = move.to_tier
                break
        return new_containers

    def solve(
        self,
        initial_containers: List[Container],
        max_moves: int = 100,
    ) -> Tuple[List[Container], List[Move]]:
        """Repeatedly execute the highest-scoring feasible, unvisited move."""
        if max_moves < 0:
            raise ValueError("max_moves must be nonnegative.")

        validate_configuration(initial_containers, self.stacks, self.tiers)
        containers = copy_containers(initial_containers)
        self.move_history = []
        self.execution_log = []
        visited = {configuration_key(containers)}

        for iteration in range(max_moves + 1):
            violations = self.count_priority_violations(containers)
            if violations == 0:
                self.execution_log.append(
                    f"Iteration {iteration}: No priority violations remain."
                )
                break

            if iteration == max_moves:
                self.execution_log.append(
                    f"Move limit reached with {violations} priority violation(s) remaining."
                )
                break

            valid_moves = self.generate_valid_moves(containers)
            unvisited_moves = []
            for move in valid_moves:
                resulting_key = configuration_key(
                    self.execute_move(containers, move)
                )
                if resulting_key not in visited:
                    unvisited_moves.append(move)

            if not unvisited_moves:
                self.execution_log.append(
                    f"Iteration {iteration}: No feasible unvisited moves remain."
                )
                break

            best_move = max(unvisited_moves, key=lambda candidate: candidate.score)
            containers = self.execute_move(containers, best_move)
            self.move_history.append(best_move)
            visited.add(configuration_key(containers))

            new_violations = self.count_priority_violations(containers)
            self.execution_log.append(
                f"Iteration {iteration}: Move {best_move.container_id} from "
                f"stack {best_move.from_stack} to stack {best_move.to_stack}; "
                f"score={best_move.score:.1f}; violations: "
                f"{violations} -> {new_violations}."
            )

        self.move_history = list(self.move_history)
        return containers, self.move_history

### State, violations, and the move score

A state records the stack and tier occupied by every container. The state is feasible when each container occupies one valid position, no two containers share a position, and every stack has consecutive occupied tiers starting at the bottom. These conditions make a top-container move physically interpretable within this simplified bay model.

Let $C$ denote the current configuration. Within a stack, list containers from bottom to top. An adjacent pair $(i,j)$, with $i$ below $j$, is a priority violation when $p_i > p_j$, where $p_i$ is the priority number of container $i$. The total number of such pairs is

$$\
V(C) = \sum_{s=0}^{S-1} \sum_{t=0}^{h_s-2}
\mathbf{1}\!\left[p_{s,t} > p_{s,t+1}\right].
$$

Here, $S$ is the number of stacks, $h_s$ is the height of stack $s$, $p_{s,t}$ is the priority of the container at tier $t$ in stack $s$, and $\mathbf{1}[\cdot]$ equals 1 when its condition is true and 0 otherwise. In the code, `count_priority_violations` computes $V(C)$. A configuration with $V(C)=0$ has no adjacent priority inversions under this model.

For a candidate move $m$, the heuristic evaluates the actual resulting configuration $C_m$. Its score is

$$\
\operatorname{score}(m) =
50\bigl(V(C)-V(C_m)\bigr)
+ 20\bigl(B(C)-B(C_m)\bigr)
+ \max\!\left(0, 15-2d-(T-1-t_m)\right)
+ 15\frac{10-p_m}{10}.
$$

In this expression, $B(C)$ is the sum of the absolute deviations from average stack height for the move's source and destination stacks; $d$ is the distance between their stack indices; $T$ is the stack capacity in tiers; $t_m$ is the moved container's source tier; and $p_m$ is its priority. The four terms respectively reward an immediate reduction in violations, improved balance, a short move from a high tier, and a higher-priority container. The constants are heuristic weights, not quantities with physical units. `calculate_move_score` implements this rule, and `solve` selects the feasible candidate with the greatest score. A high score ranks a move according to this rule; it does not prove that the move leads to a globally best plan.

### Algorithmic cost

With a limit of $M$ moves, the algorithm considers at most $S(S-1)$ source-and-destination stack pairs per iteration. Scoring each candidate examines the configuration and stack heights, so a conservative bound for this implementation is $O\!\left(M S^2 (n+ST)\right)$ time, where $n$ is the number of containers and $T$ is the tier capacity. The visited-state record and move history require space proportional to the number of visited configurations and recorded moves, with each state storing the container positions. This bound describes the implemented heuristic; it is not an optimality or solution-quality guarantee.

In [ ]:
def visualize_bay_state(
    containers: List[Container],
    stacks: int,
    tiers: int,
    title: str,
    ax=None,
):
    """Draw a bay state, with color indicating container priority."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6))

    grid = [[None for _ in range(tiers)] for _ in range(stacks)]
    for container in containers:
        grid[container.current_stack][container.current_tier] = container

    priorities = [container.priority for container in containers]
    minimum_priority = min(priorities, default=1)
    maximum_priority = max(priorities, default=1)
    if minimum_priority == maximum_priority:
        minimum_priority -= 0.5
        maximum_priority += 0.5

    colormap = plt.cm.RdYlBu_r
    normalization = Normalize(vmin=minimum_priority, vmax=maximum_priority)

    for stack in range(stacks):
        for tier in range(tiers):
            container = grid[stack][tier]
            if container is None:
                facecolor = "lightgray"
                edgecolor = "gray"
                alpha = 0.3
            else:
                facecolor = colormap(normalization(container.priority))
                edgecolor = "black"
                alpha = 1.0

            rectangle = plt.Rectangle(
                (stack * 1.2, tier * 0.8),
                1.0,
                0.6,
                facecolor=facecolor,
                edgecolor=edgecolor,
                linewidth=1.5,
                alpha=alpha,
            )
            ax.add_patch(rectangle)

            if container is not None:
                ax.text(
                    stack * 1.2 + 0.5,
                    tier * 0.8 + 0.3,
                    f"{container.id}\n(Priority {container.priority})",
                    ha="center",
                    va="center",
                    fontweight="bold",
                    fontsize=9,
                )

    ax.set_xlim(-0.15, max(0.85, stacks * 1.2 - 0.1))
    ax.set_ylim(-0.15, max(0.55, tiers * 0.8 - 0.1))
    ax.set_xlabel("Stack index")
    ax.set_ylabel("Tier index (0 is the bottom)")
    ax.set_title(title, fontweight="bold")
    ax.set_xticks([stack * 1.2 + 0.5 for stack in range(stacks)])
    ax.set_xticklabels([f"S{stack}" for stack in range(stacks)])
    ax.set_yticks([tier * 0.8 + 0.3 for tier in range(tiers)])
    ax.set_yticklabels([f"T{tier}" for tier in range(tiers)])
    ax.set_aspect("equal")

    color_mapping = ScalarMappable(norm=normalization, cmap=colormap)
    ax.figure.colorbar(
        color_mapping,
        ax=ax,
        label="Container priority (lower values indicate higher priority)",
    )


def visualize_algorithm_progression(
    initial_containers: List[Container],
    final_containers: List[Container],
    move_history: List[Move],
    stacks: int,
    tiers: int,
):
    """Show selected states from the recorded move sequence."""
    number_of_moves = len(move_history)
    if number_of_moves == 0:
        return

    number_of_stages = min(5, number_of_moves + 1)
    figure, axes = plt.subplots(
        1,
        number_of_stages,
        figsize=(5 * number_of_stages, 5),
        squeeze=False,
    )
    axes = axes[0]

    stage_move_indices = [
        round(stage * number_of_moves / (number_of_stages - 1))
        for stage in range(number_of_stages)
    ]
    stage_move_indices[-1] = number_of_moves

    current_containers = copy_containers(initial_containers)
    next_move_index = 0

    for stage, move_count in enumerate(stage_move_indices):
        while next_move_index < move_count:
            current_containers = ListScheduler(stacks, tiers).execute_move(
                current_containers,
                move_history[next_move_index],
            )
            next_move_index += 1

        if stage == 0:
            title = "Initial state"
        elif stage == number_of_stages - 1:
            title = f"Final state ({number_of_moves} moves)"
            current_containers = copy_containers(final_containers)
        else:
            title = f"After {move_count} moves"

        visualize_bay_state(
            current_containers,
            stacks,
            tiers,
            title,
            axes[stage],
        )

    figure.tight_layout()
    plt.show()

In [ ]:
# Define the concrete example. Tiers are indexed from the bottom, starting at 0.
print("=" * 60)
print("YARD PRE-MARSHALLING PROBLEM - LIST SCHEDULING ALGORITHM")
print("=" * 60)

containers = [
    Container("E", 5, 0, 1),
    Container("B", 2, 0, 0),
    Container("A", 1, 1, 1),
    Container("D", 4, 1, 0),
    Container("C", 3, 2, 0),
]

stacks, tiers = 3, 4
validate_configuration(containers, stacks, tiers)

print("\nProblem configuration:")
print(f"- Stacks: {stacks}")
print(f"- Tiers per stack: {tiers}")
print(f"- Containers: {len(containers)}")

print("\nInitial container configuration:")
for container in containers:
    print(
        f"  Container {container.id}: priority {container.priority}, "
        f"stack {container.current_stack}, tier {container.current_tier}"
    )

initial_violations = ListScheduler(stacks, tiers).count_priority_violations(
    containers
)
print(f"\nInitial adjacent priority violations: {initial_violations}")

visualize_bay_state(
    containers,
    stacks,
    tiers,
    "Initial bay configuration",
)

In [ ]:
scheduler = ListScheduler(stacks, tiers)

print("\n" + "=" * 60)
print("LIST SCHEDULING ALGORITHM EXECUTION")
print("=" * 60)

start_time = time.perf_counter()
final_containers, move_history = scheduler.solve(containers, max_moves=50)
execution_time = time.perf_counter() - start_time

print(f"\nElapsed execution time: {execution_time:.6f} seconds")
print(f"Total moves executed: {len(move_history)}")
print("\nExecution log:")
for log_entry in scheduler.execution_log:
    print(f"  {log_entry}")

In [ ]:
print("\n" + "=" * 60)
print("FINAL SOLUTION ANALYSIS")
print("=" * 60)

print("\nFinal container configuration:")
for container in final_containers:
    print(
        f"  Container {container.id}: priority {container.priority}, "
        f"stack {container.current_stack}, tier {container.current_tier}"
    )

final_violations = scheduler.count_priority_violations(final_containers)
print("\nSolution quality:")
print(f"  Initial adjacent priority violations: {initial_violations}")
print(f"  Final adjacent priority violations: {final_violations}")

assert len({container.id for container in final_containers}) == len(containers)
validate_configuration(final_containers, stacks, tiers)
assert len(move_history) <= 50

if final_violations == 0:
    print("  All adjacent priority violations have been eliminated.")
else:
    print(f"  The heuristic stopped with {final_violations} violation(s) remaining.")

if move_history:
    visualize_algorithm_progression(
        containers,
        final_containers,
        move_history,
        stacks,
        tiers,
    )

## Interpreting the result and the heuristic's limits

The printed move count and execution time describe this run on the concrete example; they are not general performance guarantees. The final violation count measures the adjacent inversions defined by this notebook. A zero count means that no adjacent pair in any stack is out of priority order under the model. It does not establish that the plan is optimal with respect to move count, travel distance, or any operational objective not represented in the score.

The method is a list-scheduling heuristic: it ranks the currently feasible moves and commits to the highest-scoring unvisited result. Its decisions are transparent because the score components and weights are explicit. However, an immediate score is only a local guide. A move can fail to reduce violations, and the move limit or the set of unvisited feasible moves can stop the run before all violations are removed. Results depend on the input configuration and on the chosen scoring rule.

The implementation's work grows with the number of candidate moves evaluated at each iteration and the size of the bay state. The configured move limit bounds the number of iterations, while visited-state tracking prevents repeated configurations. Larger or richer operational models require additional state and feasibility checks consistent with the same list-scheduling approach.